In [72]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor,DecisionTreeClassifier

In [73]:
df=pd.DataFrame()
df['X1'] = [1,2,3,4,5,6,6,7,9,9]
df['X2'] = [5,3,6,8,1,9,5,8,9,2]
df['target'] = [1,1,0,1,0,1,0,1,0,0]

In [74]:
df

,X1,X2,target
0,1,5,1
1,2,3,1
2,3,6,0
3,4,8,1
4,5,1,0
5,6,9,1
6,6,5,0
7,7,8,1
8,9,9,0
9,9,2,0


In [75]:
class Adaboost_from_scratch():
  def __init__(self,n_estimators,df):
    self.df=df
    self.n_estimators=n_estimators
    self.df['weights']=1/(self.df.shape[0])
    self.alphas=[]
    self.models=[]
  def update_weights(self,that_df,alpha):
    if that_df['target']==that_df['prediction']:
      return that_df['weights']*np.exp(-alpha)
    else:
      return that_df['weights']*np.exp(alpha)
  def calculate_error_sum(self,that_df):
    if that_df['target']==that_df['prediction']:
      return 0
    else:
      return that_df['weights']
  def calculate_alpha(self,that_df):
    row_errors = that_df.apply(lambda row: self.calculate_error_sum(row), axis=1)
    e = row_errors.sum() / that_df['weights'].sum()
    return 0.5*np.log((1-e)/e)
  def new_dataset(self,that_df):
    that_df['cumsum_upper'] = np.cumsum(that_df['updated_weights'])
    that_df['cumsum_lower'] = that_df['cumsum_upper'] - that_df['updated_weights']
    indices=[]
    for i in range(that_df.shape[0]):
      a=np.random.random()
      for index,row in that_df.iterrows():
        if a>=row['cumsum_lower'] and a<=row['cumsum_upper']:
          indices.append(index)
    return indices
  def fit(self):
    for i in range(self.n_estimators):
      dt=DecisionTreeClassifier(max_depth=1)
      X=self.df.iloc[:,:2].values
      y=self.df.iloc[:,2].values
      dt.fit(X,y)
      self.df['prediction']=dt.predict(X)
      alpha=self.calculate_alpha(self.df)
      self.df['updated_weights']=self.df.apply(lambda row:self.update_weights(row,alpha),axis=1)
      self.df['updated_weights']=self.df['updated_weights']/self.df['updated_weights'].sum()
      self.alphas.append(alpha)
      self.models.append(dt)
      new_df=self.df.iloc[self.new_dataset(self.df),0:4]
      self.df=new_df
  def pred(self,X_q):
    total=0
    for i in range(len(self.models)):
      if self.models[i].predict(X_q)==1:
        total+=self.alphas[i]
      else:
        total-=self.alphas[i]
    return np.sign(total)

In [76]:
ada_boost=Adaboost_from_scratch(3,df)

In [77]:
ada_boost.df

,X1,X2,target,weights
0,1,5,1,0.1
1,2,3,1,0.1
2,3,6,0,0.1
3,4,8,1,0.1
4,5,1,0,0.1
5,6,9,1,0.1
6,6,5,0,0.1
7,7,8,1,0.1
8,9,9,0,0.1
9,9,2,0,0.1


In [78]:
ada_boost.fit()

In [79]:
query=np.array([9,9]).reshape(1,2)
ada_boost.pred(query)

np.float64(-1.0)